# Loading Libraries

In [88]:
import io, os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

In [131]:
load_dotenv()

True

# Loading OpenAI model

In [134]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL")

In [135]:
open_ai_model = ChatOpenAI(model_name=OPEN_AI_MODEL, openai_api_key=OPENAI_API_KEY, temperature=0, max_tokens=1000)

# Summarize code review comments

In [92]:
SUMMMARIZER_SYSTEM_PROMPT="You are a helful assistant that summarizes code review comments. Be concise, precise, and clear. Avoid non-informative comments."

In [93]:
def summarize_review_comments(review_comments):
    response = open_ai_model([SystemMessage(content=SUMMMARIZER_SYSTEM_PROMPT),
                              HumanMessage(content=f"Review comments: {review_comments}") 
    ])

# Loading Review Details

In [94]:
def extract_pr_title_description(pr_json_data):
    json_data = json.loads(pr_json_data)
    return json_data["title"], json_data["description"]

In [95]:
def extract_pr_identifier(pr_json_data):
    json_data = json.loads(pr_json_data)
    return json_data["repo"], json_data["pull_request_id"]

In [100]:
def extract_changed_file_comment_pair(pr_json_data):
    json_data = json.loads(pr_json_data)
    changed_files = json_data["changed_files"]
    file_comment_map=dict()
    file_changes_map=dict()
    for changed_file in changed_files:
        file_name = changed_file["path"]
        comments = changed_file["review_comments"]
        if comments:
            file_specific_commentents = "\n".join ([comment["body"] for comment in comments])
            file_comment_map[file_name]=file_specific_commentents
        
        changed_code = changed_file["changed_code"]
        if changed_code and file_name in file_comment_map:
            file_changes_map[file_name]=changed_code    
             
    return file_changes_map, file_comment_map

In [101]:
def load_raw_repository(repo_path):
    absolute_paths = [
    os.path.abspath(os.path.join(repo_path, filename))
    for filename in os.listdir(repo_path)
    ]
    return absolute_paths

In [102]:
def read_pr_json(pr_json_path):
    with open(pr_json_path, "r") as f:
        return f.read()

In [104]:
code_review_path="dataset/raw/pandas/65576.json"
code_review_data=read_pr_json(code_review_path)

# Testing

In [105]:
extract_changed_file_comment_pair(code_review_data)

({'doc/source/whatsnew/v3.1.0.rst': '- Fixed bug in :func:`read_hdf` where the literal string ``"nan"`` in a string :class:`Index` was incorrectly converted to ``NaN`` on read, even when a custom ``nan_rep`` was supplied (:issue:`9604`)\n- Fixed bug in :meth:`DataFrame.to_hdf` raising ``TypeError`` when the index had a non-tick :class:`DateOffset` ``freq`` (e.g. ``DateOffset(years=1)``) (:issue:`45790`)\n- Fixed bug in :meth:`DataFrame.to_hdf` with ``format="table"`` where a :class:`TimedeltaIndex` was reconstructed as a :class:`PeriodIndex` (when ``freq`` was set) or an integer :class:`Index` (otherwise) on read-back (:issue:`21466`)\n- Fixed :meth:`DataFrame.to_hdf` and :meth:`Series.to_hdf` to round-trip a :class:`CategoricalIndex` in both ``"fixed"`` and ``"table"`` formats; previously raised ``AssertionError`` (:issue:`33909`, :issue:`16118`)\n- Fixed bug in :meth:`DataFrame.to_parquet` (``pyarrow`` engine) where a local file path was opened twice, once by pandas and again by pyar

In [44]:
extract_pr_title_description(code_review_data)

('BUG: Round-trip CategoricalIndex through HDF5',
 '## Summary\n\n- Saving a frame or series with a ``CategoricalIndex`` previously raised ``AssertionError: category`` from ``_convert_index`` for both ``"fixed"`` and ``"table"`` formats.\n- Extend the index write/read paths so the categories are persisted as metadata and the integer codes as the column data, mirroring how a ``CategoricalDtype`` data column is already stored, and reconstruct the ``CategoricalIndex`` on read. Same fix covers both formats.\n- "ordered" is added to ``IndexCol._info_fields`` so the flag round-trips alongside ``freq``/``tz``/``index_name``.\n\ncloses #33909\ncloses #16118\n\n## Test plan\n\n- [x] 12 new parametrized cases in ``test_categorical.py`` covering both formats, ordered/unordered, non-sorted categories, NaN entries, numeric categories, the columns axis, ``Series``, and append.\n- [x] ``pytest pandas/tests/io/pytables`` — 573 passed, 1 skipped.\n- [x] ``mypy pandas/io/pytables.py`` clean (the one rem

In [109]:
class ReviewComment:
    def __init__(self, pr_number, repo, pr_title, pr_description, file_path, changed_code, review_comments):
        self.pr_number=pr_number
        self.repo=repo
        self.pr_title=pr_title
        self.pr_description = pr_description
        self.file_path=file_path
        self.changed_code = changed_code
        self.review_comments = review_comments

In [124]:
raw_data_repository="dataset/raw/pandas"
processed_out_file="dataset/processed/pandas.json"

In [125]:
pr_json_files=load_raw_repository(raw_data_repository)

In [126]:
def extract_relevant_pr_info(pr_json_files):
    for json_file in pr_json_files:
        pr_json_data=read_pr_json(json_file)
        pr_title, pr_description=extract_pr_title_description(pr_json_data)
        repo, pr_number=extract_pr_identifier(pr_json_data)
        file_changes_map, file_comment_map=extract_changed_file_comment_pair(pr_json_data)
        for file_name, changed_code in file_changes_map.items():
            if file_name.endswith(".py"):
                review_comments=file_comment_map[file_name]
                yield ReviewComment(pr_number, repo, pr_title, pr_description, file_name, changed_code, review_comments)

In [127]:
def save_extracted_data(elements, out_file):
    os.makedirs(os.path.dirname(processed_out_file), exist_ok=True)
    with open(processed_out_file, "w", encoding="utf-8") as f:
        json.dump([element.__dict__ for element in elements], f, indent=2)
    

In [128]:
elements = extract_relevant_pr_info(pr_json_files)
save_extracted_data(elements, processed_out_file)